In [ ]:
import folium
from folium import plugins
import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import os as os

# =============================================================================
# 1. CARREGAR PERÍMETROS URBANOS DE TODOS OS MUNICÍPIOS
# =============================================================================

print("Carregando perímetros urbanos...")

perimetros = {
    'Londrina': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpLondrina/perimetro.GeoJSON"),
    'Maringá': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpMaringa/perimetro.GeoJSON"),
    'Cambé': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpCambe/perimetro.GeoJSON"),
    'Ibiporã': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpIbipora/perimetro.GeoJSON"),
    'Apucarana': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpApucarana/perimetro.GeoJSON"),
    'Arapongas': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpArapongas/perimetro.GeoJSON"),
    'Marialva': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpMarialva/perimetro.GeoJSON"),
    'Sarandi': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpSarandi/perimetro.GeoJSON"),
    'Rolândia': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpRolandia/perimetro.GeoJSON"),
    'Mandaguari': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpMandaguari/perimetro.GeoJSON"),
    'Jandaia do Sul': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpjSul/perimetro.GeoJSON"),
    'Cambira': gpd.read_file("/Volumes/ssd_externo/UEL DOUTORADO 2022/Artigo GEO/json dados/shapeFiles/shpCambira/perimetro.GeoJSON")
}

# Converter todos para EPSG:4326
for cidade in perimetros:
    perimetros[cidade] = perimetros[cidade].to_crs('EPSG:4326')

print(f"✓ {len(perimetros)} perímetros carregados")

# =============================================================================
# 2. BAIXAR MALHA VIÁRIA INDIVIDUALMENTE PARA CADA MUNICÍPIO
# =============================================================================

print("\nBaixando malha viária por município...")

all_edges = []

for cidade, perimetro in perimetros.items():
    print(f"\n  Processando {cidade}...")
    try:
        G = ox.graph_from_polygon(perimetro.unary_union, network_type='drive')
        gdf_edges = ox.graph_to_gdfs(G, nodes=False)
        gdf_edges['cidade'] = cidade
        all_edges.append(gdf_edges)
        print(f"    ✓ {len(gdf_edges)} segmentos viários")
    except Exception as e:
        print(f"    ✗ Erro: {e}")

# Concatenar todas as malhas viárias
gdf_all_edges = pd.concat(all_edges, ignore_index=True)

print(f"\n✓ Total de segmentos viários: {len(gdf_all_edges)}")
print("\nHierarquias encontradas:")
print(gdf_all_edges['highway'].value_counts())

# =============================================================================
# 3. CÓDIGO ORIGINAL DO SEU NOTEBOOK (SEM ALTERAÇÕES)
# =============================================================================

from pyproj import Transformer
import matplotlib.colors as mcolors
from shapely.geometry import Polygon

# Paleta tim.colors()
def tim_colors(n=1000):
    colors = [
        '#000080', '#0000CC', '#0040FF', '#0080FF', '#00BFFF', '#00FFFF',
        '#00FFBF', '#00FF80', '#00FF40', '#00FF00', '#40FF00', '#80FF00',
        '#BFFF00', '#FFFF00', '#FFBF00', '#FF8000', '#FF4000', '#FF0000'
    ]
    cmap = mcolors.LinearSegmentedColormap.from_list("tim", colors, N=n)
    return [mcolors.rgb2hex(cmap(i)) for i in np.linspace(0, 1, n)]

pal14 = tim_colors(1000)

grid_spacing = 109.45
half_spacing = grid_spacing / 2

# Carregar dados consolidados
pasta = '/Users/fjcosta/Documents/projects/Tabular/resultados_inferencia/mediana'
arquivos_csv = [
    'predicoes_unit_Sarandi.csv', 'predicoes_unit_Rolandia.csv',
    'predicoes_unit_Maringa.csv', 'predicoes_unit_Marialva.csv',
    'predicoes_unit_Mandaguari.csv', 'predicoes_unit_Londrina.csv',
    'predicoes_unit_Jandaia.csv', 'predicoes_unit_Ibipora.csv',
    'predicoes_unit_Cambira.csv', 'predicoes_unit_Cambe.csv',
    'predicoes_unit_Arapongas.csv', 'predicoes_unit_Apucarana.csv'
]

dataframes = []
for arquivo in arquivos_csv:
    df = pd.read_csv(os.path.join(pasta, arquivo))
    df['cidade'] = arquivo.replace('predicoes_unit_', '').replace('.csv', '')  
    dataframes.append(df)


df_consolidado = pd.concat(dataframes, ignore_index=True)

# Criar quadrados
def create_square_utm(utm_x, utm_y, half_size):
    return Polygon([
        (utm_x - half_size, utm_y - half_size),
        (utm_x + half_size, utm_y - half_size),
        (utm_x + half_size, utm_y + half_size),
        (utm_x - half_size, utm_y + half_size),
        (utm_x - half_size, utm_y - half_size)
    ])

utm_polygons = []
for idx, row in df_consolidado.iterrows():
    square_utm = create_square_utm(row['utm_x'], row['utm_y'], half_spacing)
    utm_polygons.append(square_utm)

print(f"Criados {len(utm_polygons)} quadrados em UTM")

# Converter para lat/lon
transformer = Transformer.from_crs("EPSG:32722", "EPSG:4326", always_xy=True)

def transform_polygon_to_latlon(polygon_utm):
    coords_utm = list(polygon_utm.exterior.coords)
    coords_latlon = []
    for x_utm, y_utm in coords_utm:
        lon, lat = transformer.transform(x_utm, y_utm)
        coords_latlon.append([lat, lon])
    return coords_latlon

latlon_polygons = []
for polygon_utm in utm_polygons:
    latlon_polygon = transform_polygon_to_latlon(polygon_utm)
    latlon_polygons.append(latlon_polygon)

print(f"Convertidos {len(latlon_polygons)} quadrados para lat/lon")

# Obter min e max
min_log_price = df_consolidado['price_predicted'].min()
max_log_price = df_consolidado['price_predicted'].max()
price_range = max_log_price - min_log_price

breaks = np.linspace(min_log_price, max_log_price, 18)
real_prices = np.exp(breaks)
labs = [f"R$ {price:,.0f}" for price in real_prices]

def get_color_from_price(price):
    normalized = (price - min_log_price) / price_range
    color_index = int(normalized * (len(pal14) - 1))
    color_index = max(0, min(color_index, len(pal14) - 1))
    return pal14[color_index]

# Criar mapa base
center_utm_x = df_consolidado['utm_x'].mean()
center_utm_y = df_consolidado['utm_y'].mean()
center_lon, center_lat = transformer.transform(center_utm_x, center_utm_y)

# m = folium.Map(
#     location=[center_lat, center_lon],
#     zoom_start=10,
#     tiles='https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}',
#     attr='Google Satellite'
# )

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=10,
    tiles=None,
    attr=''
)

# Criar GeoJSON
print("Criando GeoJSON...")

features = []
for idx in range(len(latlon_polygons)):
    polygon_coords = latlon_polygons[idx]
    price = df_consolidado.iloc[idx]['price_predicted']
    cidade_nome = df_consolidado.iloc[idx]['cidade'] 
    color = get_color_from_price(price)
    
    geojson_coords = [[[coord[1], coord[0]] for coord in polygon_coords]]
    
    feature = {
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": geojson_coords
        },
        "properties": {
            "fillColor": color,
            "color": color,
            "weight": 0,
            "fillOpacity": 0.7,
            "popup": f"Predicted Median Unit Value for Urban Land at {cidade_nome} (2024): R$ {np.exp(price):,.0f}/m²"
        }
    }
    features.append(feature)

geojson_data = {
    "type": "FeatureCollection",
    "features": features
}

print(f"✅ GeoJSON criado com {len(features)} features")

folium.GeoJson(
    geojson_data,
    style_function=lambda feature: {
        'fillColor': feature['properties']['fillColor'],
        'color': feature['properties']['color'],
        'weight': feature['properties']['weight'],
        'fillOpacity': feature['properties']['fillOpacity'],
        'opacity': 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['popup'],
        aliases=[''],
        labels=False
    ),
    name='Median Value Heatmap'  
).add_to(m)


print("✅ GeoJSON adicionado ao mapa!")

# =============================================================================
# 4. ADICIONAR PERÍMETROS URBANOS
# =============================================================================

all_perimetros = gpd.GeoDataFrame(pd.concat(perimetros.values(), ignore_index=True))

folium.GeoJson(
    all_perimetros,
    style_function=lambda x: {
        'fillColor': 'none',
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0
    },
    name='Urban Perimeters'
).add_to(m)

# =============================================================================
# 5. ADICIONAR MALHA VIÁRIA HIERARQUIZADA
# =============================================================================

print("Adicionando malha viária hierarquizada...")

highway_styles = {
    'motorway': {'color': '#4B0082', 'weight': 3.5, 'opacity': 0.9},      # índigo
    'trunk': {'color': '#6A0DAD', 'weight': 3, 'opacity': 0.9},           # roxo escuro
    'primary': {'color': '#8B008B', 'weight': 2.5, 'opacity': 0.9},       # magenta escuro
    'secondary': {'color': '#9932CC', 'weight': 2, 'opacity': 0.9},       # orquídea escura
    'tertiary': {'color': '#BA55D3', 'weight': 1.5, 'opacity': 0.9},      # orquídea média
    'unclassified': {'color': '#DA70D6', 'weight': 1, 'opacity': 0.5},    # orquídea
    'residential': {'color': '#DDA0DD', 'weight': 0.5, 'opacity': 0.9},   # plum
    'service': {'color': '#EE82EE', 'weight': 0.5, 'opacity': 0.9}        # violeta
}


# No loop que adiciona as vias, mude esta parte:

for highway_type, style in highway_styles.items():
    subset = gdf_all_edges[gdf_all_edges['highway'] == highway_type]
    
    if len(subset) > 0:
        folium.GeoJson(
            subset,
            style_function=lambda x, s=style: {
                'color': s['color'],
                'weight': s['weight'],
                'opacity': s['opacity']
            },
            name=highway_type.capitalize()  
        ).add_to(m)
        
        print(f"  ✓ {highway_type}: {len(subset)} vias")



# =============================================================================
# 6. LEGENDA ORIGINAL
# =============================================================================



legend_html = f"""
<div style="position: fixed; 
           top: 10px; right: 10px; width: 280px; height: 500px; 
           background-color: white; border:2px solid grey; border-radius: 10px; z-index:9999; 
           font-size:12px; padding: 10px; overflow-y: scroll;">
<p><b>Predicted Median Unit Value for Urban Land (R$/m²: 2024)</b></p>
<table style="font-size:10px; border-collapse: collapse;">
"""

for i in range(len(breaks)-1):
    color_index = int((i / (len(breaks)-2)) * (len(pal14)-1))
    color = pal14[color_index]
    price_min = np.exp(breaks[i])
    price_max = np.exp(breaks[i+1])
    
    legend_html += f"""
    <tr>
        <td style="background-color:{color}; width:20px; height:10px; border:1px solid #ccc;"></td>
        <td style="padding-left:8px; vertical-align:middle;">R$ {price_min:,.0f}/m² - R$ {price_max:,.0f}/m²</td>
    </tr>
    """

legend_html += f"""
</table>
<div style="font-size:10px; margin-top:15px; color:#666; line-height:1.4;">
<p style="margin:0; font-weight:bold;">Paradigm:</p>
<ul style="margin:5px 0; padding-left:15px;">
    <li>Spatial Resolution: {grid_spacing:.1f}m × {grid_spacing:.1f}m</li>
    <li>Urban vacant parcel with paradigm area of 363m²</li>
</ul>
</div>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))


# =============================================================================
# 7. LEGENDA ESTRUTURA VIÁRIA
# =============================================================================


vias_legend_html = """
<div style="position: fixed; 
           top: 10px; left: 10px; width: 250px; height: auto; 
           background-color: white; border:2px solid grey; border-radius: 10px; z-index:9999; 
           font-size:12px; padding: 10px;">
<p style="margin:0 0 10px 0; font-weight:bold;">Urban Transportation Network Hierarchy</p>
<table style="font-size:11px; border-collapse: collapse; width:100%;">
"""

# Adicionar cada tipo de via
for highway_type, style in highway_styles.items():
    subset = gdf_all_edges[gdf_all_edges['highway'] == highway_type]
    if len(subset) > 0:
        vias_legend_html += f"""
        <tr>
            <td style="width:30px; height:3px; background-color:{style['color']}; border:1px solid #ccc;"></td>
            <td style="padding-left:8px; vertical-align:middle;">{highway_type.capitalize()}</td>
        </tr>
        """

vias_legend_html += """
</table>
</div>
"""

m.get_root().html.add_child(folium.Element(vias_legend_html))


# Caixa de seleção
# folium.LayerControl(position='topleft', collapsed=False).add_to(m)
# titulo_vias = """
# <div style="position: fixed; 
#            top: 10px; left: 10px; width: 250px; height: 30px; 
#            background-color: white; border:2px solid grey; border-radius: 10px 10px 0 0; 
#            z-index:9999; font-size:14px; padding: 5px 10px; font-weight:bold;">
# Road Network Hierarchy
# </div>
# """

# m.get_root().html.add_child(folium.Element(titulo_vias))

# =============================================================================
# 8. SALVAR
# =============================================================================

output_path = '/Users/fjcosta/Documents/landCoverlandValue/landvaue_transport/LandValue_TabPFN_TransportStructure.html'
m.save(output_path)

print(f"\n✓ Mapa salvo em: {output_path}")

Carregando perímetros urbanos...
✓ 12 perímetros carregados

Baixando malha viária por município...

  Processando Londrina...


/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_65767/2576129649.py:47: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  G = ox.graph_from_polygon(perimetro.unary_union, network_type='drive')


    ✓ 35101 segmentos viários

  Processando Maringá...


/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_65767/2576129649.py:47: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  G = ox.graph_from_polygon(perimetro.unary_union, network_type='drive')


    ✓ 26877 segmentos viários

  Processando Cambé...


...

    ✓ 2694 segmentos viários

  Processando Cambira...
    ✓ 916 segmentos viários

✓ Total de segmentos viários: 128319

Hierarquias encontradas:
highway
residential                     101739
tertiary                         10767
secondary                         5961
unclassified                      3831
primary                           1760
trunk                             1180
living_street                      915
secondary_link                     463
tertiary_link                      424
trunk_link                         368
motorway                           297
motorway_link                      264
primary_link                       209
[unclassified, residential]         66
busway                              44
[tertiary, residential]             13
[tertiary, unclassified]             6
[living_street, residential]         4
[motorway, motorway_link]            3
[tertiary, motorway_link]            2
[motorway_link, secondary]           1
[primary, trunk]          

/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_65767/2576129649.py:47: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  G = ox.graph_from_polygon(perimetro.unary_union, network_type='drive')


Criados 73399 quadrados em UTM
Convertidos 73399 quadrados para lat/lon
Criando GeoJSON...
✅ GeoJSON criado com 73399 features
✅ GeoJSON adicionado ao mapa!
Adicionando malha viária hierarquizada...


/var/folders/5z/5t77xq511rx6jf39mc48mzx40000gn/T/ipykernel_65767/2576129649.py:233: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_perimetros = gpd.GeoDataFrame(pd.concat(perimetros.values(), ignore_index=True))


  ✓ motorway: 297 vias
  ✓ trunk: 1180 vias
  ✓ primary: 1760 vias
  ✓ secondary: 5961 vias
  ✓ tertiary: 10767 vias
  ✓ unclassified: 3831 vias
  ✓ residential: 101739 vias

✓ Mapa salvo em: mapa_valores_vias_regiao_teste.html
